# 20 · Vendor agent SDKs, LangGraph or a hand loop: how to choose
Level L4 · Part 4 · core · ~15 min · builds on 19

## Goal
Run the same order question three ways: a hand loop, a vendor SDK's own loop and a LangGraph graph. Then pick one for
the support desk and give the reasons, using five questions instead of the name that is trending this month.

## Why it matters
Without a rule for choosing, a team picks the popular framework, builds the whole agent in it, and finds out later
that the approval pause or the model lock-in is in the wrong place. The rewrite then costs more than the first build.
Watch the three ways in steps 3 to 5 below and see what each one gives you and what it leaves out: a saved state
or a pause, and what you must write yourself.

**What it makes easy:** a hand loop is a few lines and you see every step. **What it costs:** you write the stop
rules, the saved state, the pause and the trace yourself. **When you don't need a framework:** a fixed sequence of
steps (lesson 11), or one model call with no tools.

## The idea
An agent framework is a set of choices about who runs the loop, where the state lives and what a pause does. A **hand
loop** is the code of lesson 06: you run the tools and decide when to stop. A **vendor agent SDK** is the model
vendor's own loop: it runs the tools for you. **LangGraph** is a graph of named steps with a saved state, so a run
can pause and resume. None of them makes the model smarter. Each one moves work from your code into its code.

Analogy: a kitchen. A hand loop is a knife and a board: you do every cut. A vendor SDK is a ready-meal kit from one
supplier: fast, and it suits that supplier's ingredients best. LangGraph is a commercial kitchen with stations and
tickets: more setup, but an order can wait for a manager's sign-off and pick up again tomorrow.

## Picture
```mermaid
flowchart TD
  Q["Pick one: the first three of five questions"] --> P{"Must a run pause and resume, e.g. for an approval?"}
  P -->|"yes"| G["LangGraph (saved state, interrupts)"]
  P -->|"no"| V{"Is one model vendor fixed, and is its SDK the one you want?"}
  V -->|"yes"| S["Vendor agent SDK (its loop, its tools)"]
  V -->|"no, or you need every step in your code"| H["Hand loop (the loop you own)"]
```

## Step by step
**Step 1 · the desk (setup, no model call).** The desk is a small file, `data/20/desk.py` in the topic folder, that
the next cell writes. Paths are relative to the notebook folder (so the file is `../data/20/desk.py` from there), and
`%%writefile` saves the cell below as that file. It holds fictional orders (made up for this course, as in lessons 06 to 19), one
read-only tool and the question. All three ways use the same tool, so only the way around it differs.

In [1]:
from pathlib import Path

Path("../data/20").mkdir(parents=True, exist_ok=True)  # this lesson's run-time folder

The desk file, written by the cell below. Read it once. The tool is a plain Python function with a docstring and type
hints: the hand loop and the vendor SDK both turn it into the declaration the model reads.

In [2]:
%%writefile ../data/20/desk.py
"""The support desk of lesson 20: fictional orders and one read-only tool. No model call here."""

ORDERS = {
    "1093": {"status": "in transit", "days_late": 3},
    "2207": {"status": "delivered", "days_late": 0},
}
QUESTION = "My order 1093 was due last week. Where is it?"
SYSTEM = ("You are the Brightside Outfitters support desk. Use lookup_order for order status. "
          "Answer in one sentence and name the status.")


def lookup_order(order_id: str) -> dict:
    """Look up the status of one order by its number."""
    return ORDERS.get(order_id, {"error": f"no order with id {order_id}"})

Overwriting ../data/20/desk.py


The desk is ready. The next cell imports it and checks the tool on one order. No model call yet.

In [3]:
import sys; sys.path[:0] = ["..", "../data/20"]  # the course helpers, and the desk file written above
from desk import QUESTION, SYSTEM, lookup_order

print(lookup_order("1093"))

{'status': 'in transit', 'days_late': 3}


**Step 2 · the model (setup).** The key is read from `.env`, never typed here. The model is called through the
official Google SDK, the same way as lessons 06, 10 and 19. `llm_replay` is a course helper: it saves each model call
once, and `llm_replay` replays saved calls, so no key is needed to run this notebook. This lesson makes five model
calls in total: two for the hand loop, two for the vendor SDK and one for the graph.

In [4]:
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
llm_replay.start("20_choosing-sdks")

from google import genai  # the official Google SDK
from google.genai import types
from tut_env import env  # course helper: reads a setting from the .env file (not part of the SDK)

MODEL = "gemini-3.1-flash-lite"  # the course's default model, the same as lessons 02 to 19
client = genai.Client(  # in your own project, pass your real key here
    vertexai=True, api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"))
CONFIG_HAND = types.GenerateContentConfig(  # the SDK sends the tool but does not run it: the loop is yours
    system_instruction=SYSTEM, tools=[lookup_order], temperature=0,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
CONFIG_SDK = types.GenerateContentConfig(system_instruction=SYSTEM, tools=[lookup_order], temperature=0)

LLM mode: replay (20_choosing-sdks.yaml)


**Step 3 · the hand loop (2 model calls).** You call the model, read its request, run the tool yourself, send the
result back and stop when there is no request. The state is a list you own. The step cap is a stop rule you wrote.

In [5]:
def hand_loop(question: str, max_steps: int = 3) -> str:
    history = [types.Content(role="user", parts=[types.Part(text=question)])]
    for _ in range(max_steps):  # your stop rule: a cap on steps
        turn = client.models.generate_content(model=MODEL, contents=history, config=CONFIG_HAND).candidates[0].content
        history.append(turn)
        requests = [p.function_call for p in turn.parts if p.function_call]
        if not requests:  # no tool request: the model has answered
            return " ".join(p.text for p in turn.parts if p.text)
        results = [types.Part.from_function_response(name=r.name, response=lookup_order(**r.args)) for r in requests]
        history.append(types.Content(role="user", parts=results))  # the tool result goes back to the model
    return "stopped: step cap reached"


print("hand loop:", hand_loop(QUESTION))

hand loop: Your order 1093 is currently in transit and is three days late.


**Step 4 · the vendor SDK's own loop (2 model calls).** Here the vendor is Google, so this is the Gemini SDK's own
loop. The SDK runs the tool itself, sends the result back and returns the answer, so you write no loop. The chat
object keeps the history for you. The OpenAI Agents SDK, the Claude Agent SDK and Google ADK run their own loops too:
their names are in the section "The other vendor SDKs, in words" below, and none of these three is run in this lesson.

In [6]:
chat = client.chats.create(model=MODEL, config=CONFIG_SDK)  # the SDK keeps the chat and runs the tool
print("vendor SDK:", chat.send_message(QUESTION).text)

vendor SDK: Your order 1093 is currently in transit and is three days late.


**Step 5 · the graph (1 model call, then it pauses).** Here the same tool is a graph. The state is one list of turns.
The `agent` node asks the model, and a conditional edge sends a tool request to the `tools` node.

In [7]:
import operator
from typing import Annotated, TypedDict

from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

# the checkpointer must be told which classes it may save; without this, saving Gemini messages fails or warns
SERDE =JsonPlusSerializer(allowed_msgpack_modules=[types.Content, types.Part, types.FunctionCall, types.FunctionResponse])


class Desk(TypedDict):  # the state: one list of turns, which every node reads and extends
    history: Annotated[list, operator.add]


def agent(state: Desk) -> dict:  # one model call; returns only the new turn
    turn = client.models.generate_content(model=MODEL, contents=state["history"], config=CONFIG_HAND)
    return {"history": [turn.candidates[0].content]}


def tools(state: Desk) -> dict:  # runs the requests the model made in its last turn
    requests = [p.function_call for p in state["history"][-1].parts if p.function_call]
    results = [types.Part.from_function_response(name=r.name, response=lookup_order(**r.args)) for r in requests]
    return {"history": [types.Content(role="user", parts=results)]}

The wiring is short: the edges, then the compile. `interrupt_before=["tools"]` stops the run before the tool runs and
saves the state. It is the compile-time form of pausing: the node has no pause code. Lesson 19 showed the in-node
form, the `interrupt()` call inside a node, which can carry a question and take the answer back. Resuming this form
takes `invoke(None, ...)`, the call lesson 19's Step 9 uses after a crash. This lesson stops after one model call,
so you see the pause and not the resume.

In [8]:
def route(state: Desk) -> str:  # a conditional edge: a tool request goes to `tools`, an answer ends the run
    return "tools" if [p for p in state["history"][-1].parts if p.function_call] else END


g = StateGraph(Desk)
g.add_node("agent", agent).add_node("tools", tools)
g.add_edge(START, "agent").add_conditional_edges("agent", route, ["tools", END]).add_edge("tools", "agent")
graph = g.compile(checkpointer=InMemorySaver(serde=SERDE), interrupt_before=["tools"])

The run below makes the one model call. The state it saves, under one thread id, is what lesson 19 resumes.

In [9]:
CFG = {"configurable": {"thread_id": "desk-1093"}}  # one thread id = one saved run
graph.invoke({"history": [types.Content(role="user", parts=[types.Part(text=QUESTION)])]}, CFG)
snap = graph.get_state(CFG)
print("paused before:", snap.next)
print("saved request, not yet run:", [p.function_call.name for p in snap.values["history"][-1].parts if p.function_call])

paused before: ('tools',)
saved request, not yet run: ['lookup_order']


**Step 6 · what each way gave you (no model call).** The same question, three ways. Compare what you wrote and what
you got for free. The model calls are the recorded ones: 2 for the hand loop, 2 for the vendor SDK and 1 for the graph.

In [10]:
rows = [
    ("hand loop", "you wrote the loop, the stop rule and the state", "a list in a variable", "no"),
    ("vendor SDK", "the SDK ran the tool and wrote the answer", "the SDK's chat object", "no"),
    ("LangGraph", "you wrote the nodes and the edge rules", "saved after each node", "yes: stops before the tool runs"),
]
print(f"{'way':<12}{'who runs the loop':<52}{'where the state lives':<26}{'pause?'}")
for name, who, where, pause in rows:
    print(f"{name:<12}{who:<52}{where:<26}{pause}")
print("model calls (recorded): hand loop 2, vendor SDK 2, LangGraph 1, total 5")

way         who runs the loop                                   where the state lives     pause?
hand loop   you wrote the loop, the stop rule and the state     a list in a variable      no
vendor SDK  the SDK ran the tool and wrote the answer           the SDK's chat object     no
LangGraph   you wrote the nodes and the edge rules              saved after each node     yes: stops before the tool runs
model calls (recorded): hand loop 2, vendor SDK 2, LangGraph 1, total 5


## At work
- Teams keep a hand loop for small agents and move to a graph when a run must pause or survive a restart. LangGraph
  is named in 22.6% of 1,135 agent-role job listings (March to May 2026, the course's job sample; LangChain in 34.5%).
  → lesson 14 (the approval gate) and lesson 32 (retries and idempotency).
- Whatever the choice, the same golden tasks must run on each candidate before a switch. → lesson 22 (eval depth).
- Frameworks move fast: AutoGen is in maintenance mode, so pin versions and run the eval in CI when you upgrade.
  → lesson 28 (evals in CI).

## Common mistakes
The most common one is to return a plain string from a tool. The model needs a dictionary as
the result, so the call fails before anything is sent. Run the cell to see the real error.

In [11]:
types.Part.from_function_response(name="lookup_order", response="in transit")  # wrong: a string, not a dict

ValidationError: 1 validation error for FunctionResponse
response
  Input should be a valid dictionary [type=dict_type, input_value='in transit', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/dict_type

The fix is to wrap the result in a dictionary, as `lookup_order` does: `{"status": "in transit"}`.

## Five questions for the choice
Work through these in order. The first answer that settles the matter ends the choice.

| # | Question | If yes | If no |
|---|---|---|---|
| 1 | Must a run pause for a person, or survive a restart? | LangGraph: saved state and interrupts | next question |
| 2 | Is one model vendor fixed, and do you want its loop and tools? | that vendor's SDK | next question |
| 3 | Must you see and change every step, e.g. a custom stop rule? | hand loop | next question |
| 4 | Does the team know a framework, and do the job posts name it? | the one the team knows | next question |
| 5 | Will the choice still be maintained in a year? | a framework with a release plan | the hand loop, pinned |

Questions 4 and 5 need human judgement, so the function covers the first three.

## The named frameworks (versions per the course research, 2026-10)
| Option | What it is | Choose it when | Watch out for |
|---|---|---|---|
| Hand loop | your code around the model SDK | few tools, every step visible | you own the state, pause and trace |
| LangGraph 1.2 | graph of steps, checkpointer, interrupts | approvals, resume after a crash | more concepts; named in 22.6% of agent job listings |
| OpenAI Agents SDK 0.24 | light agents, handoffs, guardrails, sessions | you build on OpenAI models | best on its own models |
| Claude Agent SDK 0.2 | the Claude Code agent loop as a library | coding and long-running agents on Claude | built for coding work; needs an Anthropic key |
| Google ADK 2.12 | code-first agents, A2A-native | you deploy on Google Cloud | Google Cloud is the best-supported path |

### The other vendor SDKs, in words
None of the three SDKs named here (OpenAI, Claude, ADK) runs in this lesson: the support desk uses the Gemini SDK, and this course records no OpenAI, Anthropic
or ADK call. The names below come from each SDK's current docs, so you can read them when you choose.

- **OpenAI Agents SDK** (`pip install openai-agents`, version 0.24.0). You build an `Agent` with its instructions and
  tools. A tool is a Python function marked with `function_tool`. `Runner.run(agent, input)` runs the loop, and you
  `await` it in an async function (`Runner.run_sync` is the blocking form). The answer is `result.final_output`. The
  same package has handoffs (one agent passes the work to another) and guardrails (checks on input and output).

- **Claude Agent SDK** (`pip install claude-agent-sdk`, version 0.2.168). `query(prompt=..., options=ClaudeAgentOptions(...))`
  returns an async iterator of the messages of one agent run. It runs the same agent loop as Claude Code, so
  its tools and its context handling are the coding agent's. It needs an Anthropic API key.

- **Google ADK** (`pip install google-adk`, version 2.12.0, released 2026-10-13). An `Agent` comes from
  `google.adk.agents.llm_agent` and takes a `model`, a `name`, an `instruction` and `tools` (plain Python functions
  with a docstring). `adk run <agent folder>` gives a terminal chat; `adk web` a browser chat, which the docs call
  development-only.

### The rest of the field
| Option | What it is | Choose it when | Watch out for |
|---|---|---|---|
| CrewAI 1.15 | role-based crews of agents | a quick demo of agents handing work on | less control over state |
| Microsoft Agent Framework 1.22 | .NET and Python agents, successor of Semantic Kernel and AutoGen | a Microsoft shop | newer (1.0 in April 2026) |
| AutoGen | multi-agent chat | legacy code | maintenance mode: use AG2 or Microsoft Agent Framework |

smolagents (code actions), LlamaIndex (document agents) and Strands (AWS) fit the same five questions. Each one goes
deeper than this lesson does.

## Try it
Write `choose` so it returns the way to build the support desk from three answers, using the five questions in order.
Run the test cells, then try the answer.

In [12]:
def choose(needs_pause: bool, one_vendor_fixed: bool, needs_every_step: bool) -> str:
    # your code here
    return "hand loop"


# expect: LangGraph, vendor SDK, hand loop, hand loop
for args in [(True, False, False), (False, True, False), (False, False, True), (False, False, False)]:
    print(args, "->", choose(*args))

(True, False, False) -> hand loop
(False, True, False) -> hand loop
(False, False, True) -> hand loop
(False, False, False) -> hand loop


**Answer** — try it first, then expand the hidden cell below.

In [13]:
def choose(needs_pause: bool, one_vendor_fixed: bool, needs_every_step: bool) -> str:
    if needs_pause:
        return "LangGraph"
    if one_vendor_fixed:
        return "vendor SDK"
    if needs_every_step:
        return "hand loop"
    return "hand loop"  # no reason to add a framework yet


for args in [(True, False, False), (False, True, False), (False, False, True), (False, False, False)]:
    print(args, "->", choose(*args))

(True, False, False) -> LangGraph
(False, True, False) -> vendor SDK
(False, False, True) -> hand loop
(False, False, False) -> hand loop


The recording is saved by the last cell of the notebook. Nothing else in this lesson calls the model.

In [14]:
llm_replay.stop()  # the last model call of this notebook is above; this saves the recording

## Recap
- A framework is a set of choices about who runs the loop, where the state lives and what a pause does.
- The hand loop shows every step. The vendor SDK runs the loop with its defaults. LangGraph saves state and pauses.
- Choose with the five questions, pause first. Then run the same eval on each candidate before you commit (lesson 22).

**Interview:** "How do you choose between LangGraph, a vendor agent SDK and a hand-written loop?" Strong answer: "I
start from what the run needs. A pause for approval or a restart means a graph with a checkpointer. A fixed vendor
and a short task point to that vendor's SDK. Otherwise I keep a small hand loop so every step is visible. Then I run
the same eval on each option and check that the framework is still maintained."